# 03 — Anomaly Detection Modeling
**Project:** Unmasking the Board — Behavioral Anomaly Detection in Human Chess

This notebook trains and compares **five anomaly detection models** on player-level behavioural features:

| Model | Type | Key idea |
|---|---|---|
| Z-Score Baseline | Statistical | Flag players whose max \|z-score\| is extreme |
| LOF | Density-based | Flag players whose local density differs from neighbours |
| Isolation Forest | Tree-based | Flag players that are easy to isolate in feature space |
| One-Class SVM | Kernel-based | Learn a boundary around normal-player behaviour |
| Autoencoder | Neural | Flag players whose profile can't be reconstructed accurately |

**Methodology highlights:**
- 70 / 15 / 15 train / val / test split; `StandardScaler` fit on train only (no leakage)
- Hyperparameter search driven by ROC-AUC on val-set synthetic injection
- Ground truth for evaluation created by injecting synthetic anomalies (no real labels exist)
- ROC curves, Precision-Recall curves, and a full model comparison table are produced automatically

In [ ]:
from pathlib import Path
import sys

_root = Path.cwd().resolve()
if not (_root / 'src').is_dir():
    _root = (_root / '..').resolve()
sys.path.insert(0, str(_root))

RESULTS_DIR = _root / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_curve, precision_recall_curve, auc

from src.data_loader import load_and_prepare
from src.features import aggregate_player_stats, add_engineered_features, get_feature_matrix
from src.models import (
    ZScoreBaseline, LOFDetector,
    IsolationForestDetector, OneClassSVMDetector, AutoencoderDetector,
    run_all_models, run_hyperparameter_search,
)
from src.validation import (
    inject_synthetic_anomalies, evaluate_injection_recovery,
    test_anomaly_vs_normal, compute_silhouette, compute_davies_bouldin,
    cross_validate_anomaly_models,
)

sns.set_theme(style='whitegrid', palette='muted')
SEED = 42

# Consistent colours for every plot in this notebook
MODEL_COLORS = {
    'ZScoreBaseline':  '#636EFA',
    'LOF':             '#EF553B',
    'IsolationForest': '#00CC96',
    'OneClassSVM':     '#AB63FA',
    'Autoencoder':     '#FFA15A',
}
MODEL_ORDER = list(MODEL_COLORS)

## 1. Data Preparation

Features are aggregated to the player level (one row per player).
`fit_scaler=False` returns raw (unscaled) values so the `StandardScaler`
can be fit **exclusively on the training partition** in the next cell —
fitting it on the full dataset would let test-set statistics influence
normalisation, which is data leakage.

In [ ]:
game_df, player_df = load_and_prepare()
agg = aggregate_player_stats(player_df)
agg = add_engineered_features(agg)

# fit_scaler=False: raw features returned; scaler applied after splitting
X_raw, meta, _ = get_feature_matrix(agg, use_acpl=False, time_control='blitz', fit_scaler=False)
feature_names = list(X_raw.columns)
X_arr = X_raw.values

print(f'Players: {X_arr.shape[0]}   Features: {X_arr.shape[1]}')
X_raw.describe()

### 1a. Train / Val / Test Split  
**70 / 15 / 15** split performed on raw data.  
- `StandardScaler.fit_transform` on **train only** — val and test receive `.transform` (same parameters, no re-learning).  
- Validation set drives hyperparameter search. Test set is touched once at final evaluation.  
- This mirrors the `src/pipeline.py` design exactly.

In [ ]:
train_idx, temp_idx = train_test_split(np.arange(len(X_arr)), test_size=0.30, random_state=SEED)
val_idx,   test_idx = train_test_split(temp_idx,               test_size=0.50, random_state=SEED)

scaler  = StandardScaler()
X_train = scaler.fit_transform(X_arr[train_idx])   # learns mean/std from train only
X_val   = scaler.transform(X_arr[val_idx])          # same transform, no re-learning
X_test  = scaler.transform(X_arr[test_idx])         # untouched until final evaluation

meta_train = meta.iloc[train_idx].reset_index(drop=True)
meta_val   = meta.iloc[val_idx].reset_index(drop=True)
meta_test  = meta.iloc[test_idx].reset_index(drop=True)

print(f'Train: {len(X_train)}  |  Val: {len(X_val)}  |  Test: {len(X_test)}')

## 2. Hyperparameter Search

Random search over each model's parameter space using **val-set synthetic injection ROC-AUC** as
the objective.  50 synthetic anomalies are injected into `X_val`; models are trained on `X_train`
and scored on `X_val + injected`, so the test set is never seen here.

- **IF / OC-SVM / LOF**: 10 random draws each over contamination, structural params (n_estimators, n_neighbors, kernel…)  
- **Autoencoder**: exhaustive mini-grid over `encoding_dim × threshold_percentile` with 30-epoch trials

In [ ]:
X_inj_tune, y_inj_tune = inject_synthetic_anomalies(
    pd.DataFrame(X_val, columns=feature_names), n=50, strategy='subtle'
)
X_inj_tune_arr = X_inj_tune if isinstance(X_inj_tune, np.ndarray) else X_inj_tune

search_df, best_params = run_hyperparameter_search(
    X_train=X_train,
    X_val_injected=X_inj_tune_arr,
    y_val_injected=y_inj_tune,
    n_iter=10,   # lighter budget for interactive use; pipeline uses 20
)

search_df.to_csv(str(RESULTS_DIR / 'hyperparameter_tuning.csv'), index=False)

print('Best params per model:')
for name, params in best_params.items():
    print(f'  {name}: {params}')

## 3. Feature Correlation Heatmap

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(
    pd.DataFrame(X_train, columns=feature_names).corr(),
    annot=True, fmt='.2f', cmap='coolwarm', center=0, vmin=-1, vmax=1,
)
plt.title('Feature Correlation Matrix (training set)')
plt.tight_layout()
plt.savefig(str(RESULTS_DIR / 'feature_correlation.png'), dpi=150)
plt.show()

## 4. Train All Five Models

Each model is built with its **tuned hyperparameters** and fit on `X_train`.
Fitted model objects are kept in `fitted_models` for reuse across all evaluation sections.

In [ ]:
model_configs = [
    ('ZScoreBaseline',  ZScoreBaseline(contamination=0.05)),
    ('LOF',             LOFDetector(**best_params.get('LOF',            {'contamination': 0.05}))),
    ('IsolationForest', IsolationForestDetector(**best_params.get('IsolationForest', {'contamination': 0.05}))),
    ('OneClassSVM',     OneClassSVMDetector(**best_params.get('OneClassSVM',   {'nu': 0.05}))),
    ('Autoencoder',     AutoencoderDetector(input_dim=X_train.shape[1], **best_params.get('Autoencoder', {}))),
]

fitted_models = {}
for name, model in model_configs:
    print(f'Fitting {name}...')
    model.fit(X_train)
    fitted_models[name] = model

# run_all_models gives us a results DataFrame with scores + ensemble vote
results = run_all_models(X_train, meta_train, model_params=best_params)
print(f'\nEnsemble anomalies flagged (≥2/3 advanced models): {results["ensemble_anomaly"].sum()}')
results.head()

## 5. ROC Curves

Since there are no real cheating labels, ground truth is created by injecting 50 synthetic anomalies
into the **test set** (the 15 % held out from training and hyperparameter search).  
Each model scores every player in this augmented set; `roc_curve` uses the injected labels.

This is principled: we control exactly which rows are anomalous, so the AUC measures how well
each model separates injected anomalies from real players.

In [ ]:
# Inject anomalies into test set — only done once, shared across ROC + PR + comparison table
X_inj_test, y_inj_test = inject_synthetic_anomalies(
    pd.DataFrame(X_test, columns=feature_names), n=50, strategy='subtle'
)
X_inj_test_arr = X_inj_test if isinstance(X_inj_test, np.ndarray) else X_inj_test

# Score every model on the injected test set
model_test_scores = {
    name: model.score(X_inj_test_arr)
    for name, model in fitted_models.items()
}

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5, label='Random baseline (AUC = 0.50)')

for name in MODEL_ORDER:
    scores = model_test_scores[name]
    scores_norm = (scores - scores.min()) / (scores.max() - scores.min() + 1e-9)
    fpr, tpr, _ = roc_curve(y_inj_test, scores_norm)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, lw=2, color=MODEL_COLORS[name], label=f'{name}  (AUC = {roc_auc:.3f})')

ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves — All Models\n(Synthetic injection, test set, subtle strategy)', fontsize=13)
ax.legend(loc='lower right', fontsize=10)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])
plt.tight_layout()
plt.savefig(str(RESULTS_DIR / 'roc_curves.png'), dpi=150)
plt.show()

## 6. Precision-Recall Curves

ROC curves can be optimistic when the anomaly class is small.  
Precision-Recall curves show the same models from a **recall vs. precision** perspective,
which is more informative when we care about catching as many real anomalies as possible
while controlling false positives.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

# Baseline: random classifier at the positive class prevalence
prevalence = y_inj_test.mean()
ax.axhline(prevalence, color='k', lw=1, linestyle='--', alpha=0.5,
           label=f'Random baseline (AP = {prevalence:.2f})')

for name in MODEL_ORDER:
    scores = model_test_scores[name]
    scores_norm = (scores - scores.min()) / (scores.max() - scores.min() + 1e-9)
    precision, recall, _ = precision_recall_curve(y_inj_test, scores_norm)
    ap = auc(recall, precision)
    ax.plot(recall, precision, lw=2, color=MODEL_COLORS[name], label=f'{name}  (AP = {ap:.3f})')

ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title('Precision-Recall Curves — All Models\n(Synthetic injection, test set, subtle strategy)', fontsize=13)
ax.legend(loc='upper right', fontsize=10)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])
plt.tight_layout()
plt.savefig(str(RESULTS_DIR / 'pr_curves.png'), dpi=150)
plt.show()

## 7. Model Comparison Table

All metrics are computed on the **held-out test set** using synthetic injection ground truth.
CV mean ± std comes from the `cv_summary.csv` produced by `src/pipeline.py`
(run `make pipeline` or `python -m src.pipeline` to generate it).

In [ ]:
# ── Test-set metrics for every model ────────────────────────────────────────
comparison_rows = []
for name in MODEL_ORDER:
    model = fitted_models[name]
    m = evaluate_injection_recovery(model, X_inj_test_arr, y_inj_test)

    # Clustering quality on the training set (unsupervised, no injection needed)
    train_labels = model.predict(X_train)
    sil = compute_silhouette(X_train, train_labels)
    db  = compute_davies_bouldin(X_train, train_labels)

    comparison_rows.append({
        'Model':           name,
        'ROC-AUC':         round(m['roc_auc'],           3),
        'Avg Precision':   round(m['average_precision'],  3),
        'Precision@k':     round(m['precision_at_k'],     3),
        'Recall@k':        round(m['recall_at_k'],        3),
        'Silhouette↑':     round(sil, 3) if not (sil != sil) else 'n/a',
        'Davies-Bouldin↓': round(db,  3) if not (db  != db)  else 'n/a',
    })

comparison_df = pd.DataFrame(comparison_rows).set_index('Model')
comparison_df.to_csv(str(RESULTS_DIR / 'model_comparison_table.csv'))

print('=== Test-Set Performance (subtle injection, 50 synthetic anomalies) ===')
display(comparison_df)

# ── CV mean ± std (from pipeline run) ───────────────────────────────────────
cv_csv = RESULTS_DIR / 'cv_summary.csv'
if cv_csv.exists():
    cv_df = pd.read_csv(cv_csv)
    display_cols = ['model', 'roc_auc_mean', 'roc_auc_std', 'roc_auc_ci95',
                    'average_precision_mean', 'average_precision_std']
    cv_show = cv_df[[c for c in display_cols if c in cv_df.columns]].copy()
    cv_show.columns = [c.replace('_', ' ').title() for c in cv_show.columns]
    print('\n=== 5-Fold CV Stability (development set, mean ± std) ===')
    display(cv_show.set_index('Model'))
else:
    print('\nCV summary not found — run `python -m src.pipeline` to generate cv_summary.csv')

## 8. Anomaly Score Distributions

Score distributions on the training set split by each model's own label.
Well-separated distributions indicate the model has found a meaningful boundary.

In [ ]:
score_cols = [c for c in results.columns if c.endswith('_score')]
fig, axes = plt.subplots(1, len(score_cols), figsize=(14, 4))

for ax, col in zip(axes, score_cols):
    model_name = col.replace('_score', '')
    label_col  = f'{model_name}_label'
    color = MODEL_COLORS.get(model_name, 'grey')
    for label, alpha, lname in [(-1, 0.85, 'Anomaly'), (1, 0.5, 'Normal')]:
        mask = results[label_col] == label
        ax.hist(results.loc[mask, col], bins=30, alpha=alpha,
                color=color if lname == 'Anomaly' else 'steelblue',
                label=lname, density=True)
    ax.set_title(model_name, fontsize=10)
    ax.set_xlabel('Anomaly Score')
    ax.legend(fontsize=8)

fig.suptitle('Anomaly Score Distributions (training set)', y=1.02, fontsize=12)
plt.tight_layout()
plt.savefig(str(RESULTS_DIR / 'model_score_distributions.png'), dpi=150)
plt.show()

## 9. Ensemble Vote Distribution and Top Flagged Players

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
vote_counts = results['anomaly_votes'].value_counts().sort_index()
ax.bar(vote_counts.index, vote_counts.values, color='steelblue', edgecolor='white')
ax.set_xlabel('Number of models flagging as anomaly (out of 3 advanced)')
ax.set_ylabel('Number of players')
ax.set_title('Ensemble Vote Distribution')
ax.set_xticks(vote_counts.index)
plt.tight_layout()
plt.savefig(str(RESULTS_DIR / 'ensemble_votes.png'), dpi=150)
plt.show()

consensus = results[results['anomaly_votes'] == 3].sort_values('IsolationForest_score', ascending=False)
print(f'Players flagged by all 3 advanced models: {len(consensus)}')
consensus[['player_id', 'avg_rating', 'n_games', 'anomaly_votes']].head(20)

## 10. Injection Recovery — All Models

Synthetic anomalies are injected using two strategies:
- `engine_perfect` — clearly anomalous players at the 99th-percentile feature values  
- `subtle` — perturbs 1/3 of features by ~1.5 SD (mimics Smurf-style manipulation)

This gives a direct performance comparison across all five detectors on the **test set**.

In [ ]:
recovery_rows = []
for strategy in ('engine_perfect', 'subtle'):
    X_inj_s, y_inj_s = inject_synthetic_anomalies(
        pd.DataFrame(X_test, columns=feature_names), n=50, strategy=strategy
    )
    X_inj_s_arr = X_inj_s if isinstance(X_inj_s, np.ndarray) else X_inj_s
    for name, model in fitted_models.items():
        m = evaluate_injection_recovery(model, X_inj_s_arr, y_inj_s)
        recovery_rows.append({'Model': name, 'Strategy': strategy, **m})

recovery_df = pd.DataFrame(recovery_rows)
for strategy in ('engine_perfect', 'subtle'):
    print(f'\n--- Strategy: {strategy} ---')
    sub = recovery_df[recovery_df['Strategy'] == strategy][['Model', 'roc_auc', 'precision_at_k', 'recall_at_k', 'average_precision']]
    display(sub.set_index('Model').sort_values('roc_auc', ascending=False).round(3))

## 11. Statistical Divergence Tests

Welch t-tests compare the feature distributions of players flagged as anomalies
against normal players (using Isolation Forest labels on the training set).
Significant features tell us *which behavioural signals drive the detections*.

In [ ]:
if_model   = fitted_models['IsolationForest']
if_labels  = if_model.predict(X_train)

divergence_df = test_anomaly_vs_normal(
    anomaly_scores=if_model.score(X_train),
    labels=if_labels,
    feature_matrix=X_train,
    feature_names=feature_names,
)
print('Welch t-tests — anomaly vs. normal players (IsolationForest labels):')
display(divergence_df)

In [ ]:
sil = compute_silhouette(X_train, if_labels)
db  = compute_davies_bouldin(X_train, if_labels)
print(f'Isolation Forest — training-set clustering quality:')
print(f'  Silhouette Score    : {sil:.4f}  (↑ better, max=1)')
print(f'  Davies-Bouldin Index: {db:.4f}  (↓ better)')

## 12. Cross-Validation Stability

A single train/test split can be misleading.  5-fold CV over the development set (train + val)
estimates how much results vary across different random partitions.  
Narrow std → results are stable; wide std → the dataset is too small or the signal is weak.

Load pre-computed results from `src/pipeline.py` if available; otherwise run inline (takes ~5 min).

In [ ]:
cv_csv = RESULTS_DIR / 'cv_summary.csv'

if cv_csv.exists():
    cv_summary = pd.read_csv(cv_csv)
    print('Loaded cv_summary.csv from pipeline run.')
else:
    print('cv_summary.csv not found — running 5-fold CV inline (this may take a few minutes)...')
    dev_idx = np.concatenate([train_idx, val_idx])
    _, cv_summary = cross_validate_anomaly_models(
        X=X_arr[dev_idx],
        feature_names=feature_names,
        best_params=best_params,
        n_splits=5,
    )
    cv_summary.to_csv(str(cv_csv), index=False)

# Display ROC-AUC mean ± std per model
display_cols = ['model', 'roc_auc_mean', 'roc_auc_std', 'roc_auc_ci95',
                'average_precision_mean', 'average_precision_std']
cv_show = cv_summary[[c for c in display_cols if c in cv_summary.columns]].copy()

# Pretty-print as "mean ± std" strings
for metric in ['roc_auc', 'average_precision']:
    mean_col, std_col = f'{metric}_mean', f'{metric}_std'
    if mean_col in cv_show.columns and std_col in cv_show.columns:
        cv_show[f'{metric} (mean ± std)'] = (
            cv_show[mean_col].map('{:.3f}'.format) + ' ± ' + cv_show[std_col].map('{:.3f}'.format)
        )
        cv_show.drop(columns=[mean_col, std_col, f'{metric}_ci95'], errors='ignore', inplace=True)

print('\n5-Fold Cross-Validation Results:')
display(cv_show.set_index('model'))

# Bar chart: ROC-AUC mean ± 1 std error bar
if 'roc_auc (mean ± std)' not in cv_show.columns:  # columns may differ if already parsed
    pass
else:
    # Re-read from the full summary for the bar chart
    cv_full = pd.read_csv(cv_csv) if cv_csv.exists() else cv_summary
    fig, ax = plt.subplots(figsize=(8, 4))
    models_cv = cv_full['model'].tolist()
    means = cv_full['roc_auc_mean'].tolist()
    stds  = cv_full['roc_auc_std'].tolist()
    colors = [MODEL_COLORS.get(m, 'grey') for m in models_cv]
    bars = ax.bar(models_cv, means, yerr=stds, capsize=5, color=colors, edgecolor='white', error_kw={'lw': 2})
    ax.axhline(0.5, color='k', lw=1, linestyle='--', alpha=0.4, label='Random baseline')
    ax.set_ylabel('ROC-AUC')
    ax.set_title('5-Fold CV — ROC-AUC Mean ± Std per Model')
    ax.set_ylim([0, 1.05])
    ax.legend()
    plt.xticks(rotation=15)
    plt.tight_layout()
    plt.savefig(str(RESULTS_DIR / 'cv_roc_auc_bar.png'), dpi=150)
    plt.show()